> 📄 English: [ollama_tutorial.ipynb](ollama_tutorial.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--on-prem-llm-ollama--ollama-tutorial)

# 使用 **Ollama** 的本地大语言模型（On‑Prem LLM）

欢迎！本教程将演示如何使用 [Ollama](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--on-prem-llm-ollama--ollama-tutorial&click=ollama&target=https%3A%2F%2Follama.com&text=Ollama) **在你自己的硬件上完整运行最先进的（state‑of‑the‑art）语言模型**。

**用 Ollama 替代外部 LLM API！**

学习如何用本地 Ollama 模型替换你 AI Agent 中的云端 LLM（OpenAI、Anthropic 等）。本教程涵盖从基础 API 调用到完整迁移 Agent 的所有内容。

Ollama 非常适合入门本地 LLM，不过进阶用户之后可以探索 vLLM、TensorRT‑LLM 或自定义推理服务器等替代方案，以获得最佳性能。

---

## 概览
**Ollama** 是一个轻量级运行时，让你可以通过本地 REST API 下载、运行并交互开源权重 LLM（如 Llama 3）。整个过程**不涉及外部云服务**，你的数据永远不会离开本机。

| 章节 | 你将学到什么 |
|---------|-------------------|
| 1. 安装 | 在 macOS / Linux / Windows 或 Docker 上配置 Ollama 守护进程 |
| 2. 模型下载 | 拉取量化模型文件（`.gguf`）并启动服务器 |
| 3. 测试你的环境 | 用 `curl` 发送聊天请求并查看流式响应 |
| 4. Python API | 通过 `requests` 和 **LangChain** 的 `ChatOllama` 包装器调用 REST 端点 |
| 5. API 参数 | 用 temperature、top_p、system 提示词等参数自定义模型行为 |
| 6. 故障排查 | 解决常见问题 |
| 7. 练习 Ollama 使用场景 | 示例指南 |

---

## 动机
许多组织需要生成式 AI 能力，但**不希望把敏感 IP 发送到外部服务**。在本地（on‑prem）运行模型可以带来：

* **数据主权** – 完全掌控数据和模型权重
* **可预测的成本** – 没有按 token 计费，只有硬件资源消耗
* **低延迟** – 一切都在局域网速度下完成
* **灵活性** – 随意更换模型、调整量化，或离线微调

---

## 核心组件

1. **Ollama 守护进程** – 后台进程，负责加载模型并暴露 REST/GRPC 接口。
2. **模型文件**（`.gguf`）– 针对消费级 GPU/CPU 优化的量化权重。
3. **客户端接口** – CLI（`ollama run ...`）、REST、Python SDK、LangChain 集成。

---

## 方法细节

我们将逐步讲解：

1. **安装** – 一行脚本或 Docker 镜像。
2. **模型拉取与启动** – 获取 *Llama 3.1 8‑b* 并启动运行时。
3. **首次聊天请求** – 发送 `curl` 调用并观察 JSON 流。
4. **Python requests 客户端** – 用纯 Python 封装该调用。
5. **LangChain 集成** – 作为云端模型的直接替代（drop‑in replacement）。

---

## 优势

* ✅ 将机密数据保留在本地，适合企业环境和敏感工作负载
* ✅ 可完全离线或在隔离（air‑gapped）系统中运行，对安全或断网场景至关重要
* ✅ 安装和运行简单，无需复杂配置或依赖云服务
* ✅ 易于与现有工具集成，包括 CLI、REST API、LangChain 等
* ✅ 消除按 token 计费与 API 限制，为大规模或持续使用提供高性价比方案
* ✅ 让你完全掌控模型与配置，可按性能、大小和行为进行定制

> **Ollama 适合你吗？**
> 当你出于隐私、安全、合规要求，或为了摆脱云依赖而需要在本地运行大语言模型时，Ollama 尤其有用。无论你是在构建内部工具、为敏感工作流开发 AI Agent，还是在隔离环境中运行，Ollama 都能提供灵活可靠的本地 LLM 解决方案。

---

## 快速开始

> **📝 注意：** 本教程中的所有示例均使用 `stream: false`，以简化响应处理。你也可以试试 `stream: true`，以在生成过程中实时接收响应，这对交互式应用很有帮助，但需要处理部分（partial）响应。

### 最低硬件要求

- **内存（RAM）**：最低 8GB（较大模型建议 16GB 以上）
- **存储**：10GB 以上可用空间（模型大小从 4GB 到 70GB 以上不等）
- **CPU**：任何现代 x64 处理器（macOS 也支持 ARM64）
- **GPU**：可选但推荐（NVIDIA、AMD 或 Apple Silicon 用于加速）

---

*用* <kbd>Shift</kbd> + <kbd>Enter</kbd> *运行每个单元格，或使用工具栏中的“Run All”按钮。*


# 1. 安装
运行以下命令，为 macOS / Linux 下载并安装 Ollama 守护进程。Windows 用户可以从官方网站下载 `.exe` 文件。


> **💡 提示**：你还可以使用 Docker 容器运行 Ollama！请参阅[官方 Docker 镜像指南](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--on-prem-llm-ollama--ollama-tutorial&click=ollama-ollama-is-now-available-as-an-official-docker-image&target=https%3A%2F%2Follama.com%2Fblog%2Follama-is-now-available-as-an-official-docker-image&text=official%20Docker%20image%20guide)获取配置说明。


# 2. 下载预训练模型并启动服务器
`ollama pull` 会拉取量化权重。`ollama serve` 会启动本地 REST 服务。


> **⚠️ Windows 注意**：如果你看到“only one usage of each socket address is permitted”，说明 Ollama 很可能已经在运行了。在 Windows 上，Ollama 通常会在安装后自动启动。尝试跳过 `ollama serve`，直接进行测试。


# 3. 测试你的环境
通过 REST API 发起你的第一次聊天请求

这个 `curl` 调用会向本地服务器发送一条简单消息，并以 JSON 形式流式返回响应。


**Linux / IOS 命令：**


**Windows CMD 替代命令：**


**Postman 替代方式：**
- **方法（Method）**：POST
- **URL**：`http://localhost:11434/api/chat`
- **Headers**：`Content-Type: application/json`
- **Body**（原始 JSON）：
{
  "model": "llama3.1:8b",
  "stream": false,
  "messages": [
    {
      "role": "user",
      "content": "Hello!"
    }
  ]
}


# 4. Python API

### 替换 OpenAI API 调用

**之前（OpenAI）：**


In [ ]:
from openai import OpenAI

prompt = "Hello!"
client = OpenAI(api_key="your-key")
response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": prompt}]
)
print(response.choices[0].message.content)


**之后（Ollama）：**


In [ ]:
import requests

prompt = "Hello!"
response = requests.post("http://localhost:11434/api/chat", json={
    "model": "llama3.1:8b",
    "messages": [{"role": "user", "content": prompt}],
    "stream": False
})
data = response.json()
print(data["message"]["content"])


### 替换 LangChain 模型

**之前（OpenAI）：**


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4")
response = llm.invoke("Hello!")


**之后（Ollama）：**


In [ ]:
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(model="llama3.1:8b")
response = llm.invoke("Hello!")
print(response.content)


# 5. Ollama API 参数

Ollama 提供了丰富的自定义选项。以下是常用参数：

### 基础参数

| **参数**    | **类型** | **默认值** | **描述**                                                                              | **何时使用以及如何使用**                                                                                                                                                 |
| ---------------- | -------- | ----------- | -------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `model`          | string   | –           | **必填。** 模型标识符（例如 `"llama3.1:8b"`）。                                      | 始终必填。根据你的任务选择模型（例如，资源允许时用 `llama3.1:8b` 获得快速轻量的响应，或用 `llama3.1:70b` 获得更高质量）。 |
| `messages`       | array    | –           | 聊天历史，是一个 `{role: user/assistant/system, content: ...}` 消息列表。            | 提供上下文时必填。 |
| `stream`         | boolean  | `true`      | 是否逐 token 流式返回响应。                                               | 实时 UI 或聊天应用用 `true`。如果需要一次性处理完整响应，则用 `false`。|
| `temperature`    | float    | `0.8`       | 控制随机性。`0.0` = 确定性输出，`2.0` = 非常随机。                             |调低（`0.0–0.3`）以减少变化、让输出更可预测。当可接受多样化或探索性输出时调高（`>1.0`）。|
| `top_p`          | float    | `0.9`       | 核采样（nucleus sampling）：只从累积概率为 `top_p` 的 top tokens 中采样。       | 保守答案调低（`0.5–0.8`）；平衡输出保持 `0.9`。可与 `temperature` 搭配使用或替代它。                                                     |
| `num_predict`    | int      | `128`       | 生成的最大 token 数。`-1` 表示不限（直到 `stop` 或内部限制）。 | 答案更长就调大。简洁答案用较小值（`50–150`），长文用较大值（`256+`）。限制该值以控制延迟。                                 |
| `repeat_penalty` | float    | `1.1`       | 惩罚重复。`1.0` = 无惩罚，>1 会抑制重复的短语。                   | 模型重复内容时调到 `1.2–1.5`。自然重复（如诗歌）保持 `1.0–1.1`。                                                           |
| `system`         | string   | –           | 可选的 system 提示词，用于设定助手的行为。                                 | 用于定义语气、专业领域或任务。示例：`"You are a helpful medical assistant."`                                                                             |
| `stop`           | array    | –           | 遇到这些字符串时将停止生成的字符串列表。                                  | 用于防止回答无限延续。示例：`["\nUser:", "</s>"]` 或用于工具的自定义分隔符。                                                                       |


### 性能参数

| 参数 | 类型 | 默认值 | 描述 |
|-----------|------|---------|-------------|
| `num_ctx` | int | 2048 | 上下文窗口大小（内存中的最大 token 数） |
| `num_gpu` | int | -1 | GPU 层数（-1=自动，0=仅 CPU） |
| `keep_alive` | string | - | 保持模型加载（"5m"、"10s"、"-1"=永久） |

### 使用示例

**带常用参数的基础 API 调用：**


In [ ]:
import requests

response = requests.post("http://localhost:11434/api/chat", json={
	"model": "llama3.1:8b",
	"messages": [{"role": "user", "content": "Explain quantum computing"}],
	"stream": False,
	"temperature": 0.3,  # Lower for factual responses
	"top_p": 0.9,  # Nucleus sampling
	"num_predict": 500,  # Limit response length
	"repeat_penalty": 1.2,  # Reduce repetition
	"stop": ["```", "---"]  # Stop at code blocks or separators
})
data = response.json()

print(data["message"]["content"])


**与 LangChain 一起使用：**


In [ ]:
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(
	model="llama3.1:8b",
	temperature=0.7,
	top_p=0.9,
	num_predict=256,
	repeat_penalty=1.1
)
response = llm.invoke("Hello!")
print(response.content)


> **💡 提示：** 使用 `stream: false` 可以简化处理 • 设置 `num_predict` 可限制响应长度 • 使用 `keep_alive` 可避免重复加载模型 • 调整 `temperature` 可在创造性与一致性之间权衡

> **⚠️ 注意：** 并非所有参数都适用于每个模型。较大的 `num_ctx` 值需要更多 RAM/VRAM。


# 6. 故障排查


**找不到模型？**


**连接被拒绝？**


**内存不足？**
试试更小的模型，例如 `mistral:7b`


### 模型推荐

| 模型 | 大小 | 最适合 | 速度 |
|-------|------|----------|-------|
| `llama3.1:8b` | 8GB RAM | 通用、Agent | 快 |
| `qwen2.5:14b` | 14GB RAM | 代码、推理 | 中等 |
| `phi3:14b` | 14GB RAM | 高效任务 | 快 |
| `mistral:7b` | 7GB RAM | 简单任务 | 非常快 |


# 7. 练习 Ollama 使用场景

- 从 `basic_usage.ipynb` 开始，了解简单的替换方式
- 尝试 `langchain_agent.ipynb`，了解 Agent 模式
- 针对你的使用场景实验不同的模型

准备好让你的 Agent 完全本地化了吗？从示例开始吧！
